# PlantDisease dataset inventory

Inventories only `/kaggle/input/plantdisease/PlantVillage`, writes structured outputs, and exports one sample image per subfolder to `/kaggle/working`.

In [ ]:
import csv
import json
import shutil
import zipfile
from pathlib import Path


In [ ]:
input_root = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd().parent / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

dataset_slug = "plantdisease"
dataset_root = input_root / dataset_slug
scan_root = dataset_root / "PlantVillage"

if not scan_root.exists():
    raise FileNotFoundError(f"Expected PlantVillage directory not found: {scan_root}")

print(f"Input root: {input_root}")
print(f"Dataset root: {dataset_root}")
print(f"Scan root: {scan_root}")


In [ ]:
directories = []
files = []

for path in sorted(scan_root.rglob("*")):
    rel_path = path.relative_to(scan_root).as_posix()
    if path.is_dir():
        directories.append(rel_path)
        continue

    if path.is_file():
        parts = rel_path.split("/")
        subfolder = parts[0] if len(parts) > 1 else ""
        files.append(
            {
                "subfolder": subfolder,
                "file_name": path.name,
                "relative_path": rel_path,
                "size_bytes": path.stat().st_size,
                "suffix": path.suffix.lower(),
            }
        )

print(f"Directories: {len(directories)}")
print(f"Files: {len(files)}")


In [ ]:
class_counts = {}
suffix_counts = {}
for item in files:
    class_counts[item["subfolder"]] = class_counts.get(item["subfolder"], 0) + 1
    suffix_counts[item["suffix"]] = suffix_counts.get(item["suffix"], 0) + 1

summary = {
    "input_root": str(input_root),
    "dataset_slug": dataset_slug,
    "dataset_root": str(dataset_root),
    "scan_root": str(scan_root),
    "scan_root_exists": scan_root.exists(),
    "directory_count": len(directories),
    "file_count": len(files),
    "total_size_bytes": sum(item["size_bytes"] for item in files),
    "class_count": len(class_counts),
    "class_counts": dict(sorted(class_counts.items())),
    "suffix_counts": dict(sorted(suffix_counts.items())),
    "sample_directories": directories[:20],
    "sample_files": files[:20],
}

print(f"Classes: {summary['class_count']}")
print(f"Total size bytes: {summary['total_size_bytes']}")


In [ ]:
image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
sample_images_dir = output_dir / "sample_images"
sample_images_dir.mkdir(parents=True, exist_ok=True)

sample_images = []
seen_subfolders = set()

for item in files:
    subfolder = item["subfolder"]
    if not subfolder or subfolder in seen_subfolders:
        continue
    if item["suffix"] not in image_suffixes:
        continue

    source_path = scan_root / item["relative_path"]
    output_name = f"{subfolder}{item['suffix']}"
    output_path = sample_images_dir / output_name
    shutil.copy2(source_path, output_path)

    sample_images.append(
        {
            "subfolder": subfolder,
            "source_file_name": item["file_name"],
            "source_relative_path": item["relative_path"],
            "output_file_name": output_name,
            "output_relative_path": output_path.relative_to(output_dir).as_posix(),
            "size_bytes": item["size_bytes"],
        }
    )
    seen_subfolders.add(subfolder)

zip_path = output_dir / "plantdisease_sample_images.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for image_path in sorted(sample_images_dir.iterdir()):
        if image_path.is_file():
            archive.write(image_path, image_path.relative_to(output_dir).as_posix())

summary["sample_image_count"] = len(sample_images)
summary["sample_images"] = sample_images
summary["sample_images_zip"] = zip_path.name

print(f"Sample images: {len(sample_images)}")
print(f"Wrote: {zip_path}")


In [ ]:
json_path = output_dir / "plantdisease_inventory_summary.json"
csv_path = output_dir / "plantdisease_files.csv"

json_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

csv_columns = ["subfolder", "file_name", "relative_path", "size_bytes", "suffix"]
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=csv_columns)
    writer.writeheader()
    writer.writerows(files)

print(f"Wrote: {json_path}")
print(f"Wrote: {csv_path}")
